<a href="https://colab.research.google.com/github/vanditha18/NLP-Notebooks/blob/main/Text_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Scrapping

In [ ]:
import os
import csv
import json
import requests


#get it from env variable
API_KEY = os.getenv("TMDB_API_KEY")

if not API_KEY:
    raise ValueError("Please set the TMDB_API_KEY environment variable")

#Base URL
BASE_URL = "https://api.themoviedb.org/3/movie/top_rated"

#Set up the folder to dump
ROOT_DIR = "tmdb_movie_data"
os.makedirs(ROOT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(ROOT_DIR, "movies.csv")
rows = []

#Loop through pages and dump the data to json
pg_num_start = 1
page_num_end = 30
for page in range(pg_num_start, page_num_end):
    params = {
        "api_key": API_KEY,
        "language": "en-US",
        "page": page
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        data = response.json()
        for movie_result in data['results']:
          description = movie_result.get('overview')
          genre_ids = movie_result.get('genre_ids',[])
          rows.append([description, genre_ids])

        file_path = os.path.join(ROOT_DIR, f"page{page}.json")

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)

        print(f"Saved: {file_path}")
    else:
        print(f"Failed for page {page}: {response.status_code}")


# Write directly to CSV
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["description", "genre_ids"])
    writer.writerows(rows)

print("CSV created: movies.csv")

Saved: tmdb_movie_data/page1.json
Saved: tmdb_movie_data/page2.json
Saved: tmdb_movie_data/page3.json
Saved: tmdb_movie_data/page4.json
Saved: tmdb_movie_data/page5.json
Saved: tmdb_movie_data/page6.json
Saved: tmdb_movie_data/page7.json
Saved: tmdb_movie_data/page8.json
Saved: tmdb_movie_data/page9.json
Saved: tmdb_movie_data/page10.json
Saved: tmdb_movie_data/page11.json
Saved: tmdb_movie_data/page12.json
Saved: tmdb_movie_data/page13.json
Saved: tmdb_movie_data/page14.json
Saved: tmdb_movie_data/page15.json
Saved: tmdb_movie_data/page16.json
Saved: tmdb_movie_data/page17.json
Saved: tmdb_movie_data/page18.json
Saved: tmdb_movie_data/page19.json
Saved: tmdb_movie_data/page20.json
Saved: tmdb_movie_data/page21.json
Saved: tmdb_movie_data/page22.json
Saved: tmdb_movie_data/page23.json
Saved: tmdb_movie_data/page24.json
Saved: tmdb_movie_data/page25.json
Saved: tmdb_movie_data/page26.json
Saved: tmdb_movie_data/page27.json
Saved: tmdb_movie_data/page28.json
Saved: tmdb_movie_data/page29

In [ ]:
## Map the genre ids to genres
genre_json = "/content/tmdb_movie_data/genre_id.json"
with open(genre_json, "r") as f:
  genre_data = json.load(f)['genres']
genre_data = {genre['id']: genre['name'] for genre in genre_data}

## Text preprocessing

In [ ]:
import pandas as pd
import re
import emoji
from nltk.stem.snowball import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer
import nltk
import ast
import spacy
from textblob import TextBlob

In [ ]:
df = pd.read_csv(OUTPUT_CSV)
df["genre_ids"] = df["genre_ids"].apply(ast.literal_eval)
df

,description,genre_ids
0,Imprisoned in the 1940s for the double murder ...,"[18, 80]"
1,"Spanning the years 1945 to 1955, a chronicle o...","[18, 80]"
2,In the continuing saga of the Corleone crime f...,"[18, 80]"
3,The true story of how businessman Oskar Schind...,"[18, 36, 10752]"
4,The defense and the prosecution have rested an...,[18]
...,...,...
575,A young widow flees from Rome during WWII and ...,"[18, 10752]"
576,Tom Joad returns to his home after a jail sent...,[18]
577,A photographer takes up newsreel shooting to i...,"[35, 10749]"
578,When a machine that allows therapists to enter...,"[878, 53, 16]"


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 580 entries, 0 to 579
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   description  580 non-null    object
 1   genre_ids    580 non-null    object
dtypes: object(2)
memory usage: 9.2+ KB


In [ ]:
def get_genre_names(genre_ids):
    genre_names = ""
    for genre_id in genre_ids:
        genre_name = genre_data.get(genre_id)
        genre_names += genre_name + ", "
    return genre_names.rstrip(", ")

In [ ]:
df['genres'] = df['genre_ids'].apply(get_genre_names)
df

,description,genre_ids,genres
0,Imprisoned in the 1940s for the double murder ...,"[18, 80]","Drama, Crime"
1,"Spanning the years 1945 to 1955, a chronicle o...","[18, 80]","Drama, Crime"
2,In the continuing saga of the Corleone crime f...,"[18, 80]","Drama, Crime"
3,The true story of how businessman Oskar Schind...,"[18, 36, 10752]","Drama, History, War"
4,The defense and the prosecution have rested an...,[18],Drama
...,...,...,...
575,A young widow flees from Rome during WWII and ...,"[18, 10752]","Drama, War"
576,Tom Joad returns to his home after a jail sent...,[18],Drama
577,A photographer takes up newsreel shooting to i...,"[35, 10749]","Comedy, Romance"
578,When a machine that allows therapists to enter...,"[878, 53, 16]","Science Fiction, Thriller, Animation"


### Lowercasing

In [ ]:
df['description'] = df['description'].str.lower()
df

,description,genre_ids,genres
0,imprisoned in the 1940s for the double murder ...,"[18, 80]","Drama, Crime"
1,"spanning the years 1945 to 1955, a chronicle o...","[18, 80]","Drama, Crime"
2,in the continuing saga of the corleone crime f...,"[18, 80]","Drama, Crime"
3,the true story of how businessman oskar schind...,"[18, 36, 10752]","Drama, History, War"
4,the defense and the prosecution have rested an...,[18],Drama
...,...,...,...
575,a young widow flees from rome during wwii and ...,"[18, 10752]","Drama, War"
576,tom joad returns to his home after a jail sent...,[18],Drama
577,a photographer takes up newsreel shooting to i...,"[35, 10749]","Comedy, Romance"
578,when a machine that allows therapists to enter...,"[878, 53, 16]","Science Fiction, Thriller, Animation"


### Remove HTML tags  and URL's and Punctuation marks

In [ ]:
def clean_text(text):
    # Remove HTML tags
    text = re.compile('<.*?>').sub('', text)
    # Remove URL's
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\S+', '', text)
    # Remove punctuation and digits
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    return text

In [ ]:
df['description'][10]

'as armies mass for a final battle that will decide the fate of the world--and powerful, ancient forces of light and dark compete to determine the outcome--one member of the fellowship of the ring is revealed as the noble heir to the throne of the kings of men. yet, the sole hope for triumph over evil lies with a brave hobbit, frodo, who, accompanied by his loyal friend sam and the hideous, wretched gollum, ventures deep into the very dark heart of mordor on his seemingly impossible quest to destroy the ring of power.\u200b'

In [ ]:
df['description'] = df['description'].apply(clean_text)
df

,description,genre_ids,genres
0,imprisoned in the 1940s for the double murder ...,"[18, 80]","Drama, Crime"
1,spanning the years 1945 to 1955 a chronicle of...,"[18, 80]","Drama, Crime"
2,in the continuing saga of the corleone crime f...,"[18, 80]","Drama, Crime"
3,the true story of how businessman oskar schind...,"[18, 36, 10752]","Drama, History, War"
4,the defense and the prosecution have rested an...,[18],Drama
...,...,...,...
575,a young widow flees from rome during wwii and ...,"[18, 10752]","Drama, War"
576,tom joad returns to his home after a jail sent...,[18],Drama
577,a photographer takes up newsreel shooting to i...,"[35, 10749]","Comedy, Romance"
578,when a machine that allows therapists to enter...,"[878, 53, 16]","Science Fiction, Thriller, Animation"


In [ ]:
df['description'][10]

'as armies mass for a final battle that will decide the fate of the worldand powerful ancient forces of light and dark compete to determine the outcomeone member of the fellowship of the ring is revealed as the noble heir to the throne of the kings of men yet the sole hope for triumph over evil lies with a brave hobbit frodo who accompanied by his loyal friend sam and the hideous wretched gollum ventures deep into the very dark heart of mordor on his seemingly impossible quest to destroy the ring of power'

### Chat word treatment

In [ ]:
chat_words = "/content/tmdb_movie_data/slang.txt"
with open(chat_words, "r") as f:
  chat_words = f.read()
chat_words = dict(line.split("=", 1) for line in chat_words.split("\n"))

In [ ]:
chat_words

{'A3': 'Anytime, Anywhere, Anyplace',
 'ADIH': 'Another Day In Hell',
 'AFK': 'Away From Keyboard',
 'AFAIK': 'As Far As I Know',
 'ASAP': 'As Soon As Possible',
 'ASL': 'Age, Sex, Location',
 'ATK': 'At The Keyboard',
 'ATM': 'At The Moment',
 'BAE': 'Before Anyone Else',
 'BAK': 'Back At Keyboard',
 'BBL': 'Be Back Later',
 'BBS': 'Be Back Soon',
 'BFN': 'Bye For Now',
 'B4N': 'Bye For Now',
 'BRB': 'Be Right Back',
 'BRUH': 'Bro',
 'BRT': 'Be Right There',
 'BSAAW': 'Big Smile And A Wink',
 'BTW': 'By The Way',
 'BWL': 'Bursting With Laughter',
 'CSL': 'Can’t Stop Laughing',
 'CU': 'See You',
 'CUL8R': 'See You Later',
 'CYA': 'See You',
 'DM': 'Direct Message',
 'FAQ': 'Frequently Asked Questions',
 'FC': 'Fingers Crossed',
 'FIMH': 'Forever In My Heart',
 'FOMO': 'Fear Of Missing Out',
 'FR': 'For Real',
 'FWIW': "For What It's Worth",
 'FYP': 'For You Page',
 'FYI': 'For Your Information',
 'G9': 'Genius',
 'GAL': 'Get A Life',
 'GG': 'Good Game',
 'GMTA': 'Great Minds Think Alik

In [ ]:
def chat_conversion(text):
    new_text = ""
    for w in text.split():
      if w.upper() in chat_words:
        new_text = new_text + chat_words[w.upper()] + " "
      else:
        new_text = new_text + w + " "
    return new_text

In [ ]:
df['description'] = df['description'].apply(chat_conversion)
df

,description,genre_ids,genres
0,imprisoned in the 1940s for the double murder ...,"[18, 80]","Drama, Crime"
1,spanning the years 1945 to 1955 a chronicle of...,"[18, 80]","Drama, Crime"
2,in the continuing saga of the corleone crime f...,"[18, 80]","Drama, Crime"
3,the true story of how businessman oskar schind...,"[18, 36, 10752]","Drama, History, War"
4,the defense and the prosecution have rested an...,[18],Drama
...,...,...,...
575,a young widow flees from rome during wwii and ...,"[18, 10752]","Drama, War"
576,tom joad returns to his home after a jail sent...,[18],Drama
577,a photographer takes up newsreel shooting to i...,"[35, 10749]","Comedy, Romance"
578,when a machine that allows therapists to enter...,"[878, 53, 16]","Science Fiction, Thriller, Animation"


In [ ]:
chat_conversion('Iam at home rn')

'Iam at home Right Now '

### Spelling correction

In [ ]:
df['description'] = df['description'].apply(lambda x: TextBlob(x).correct().string)
df

,description,genre_ids,genres
0,imprisoned in the 1940s for the double murder ...,"[18, 80]","Drama, Crime"
1,spanning the years 1945 to 1955 a chronicle of...,"[18, 80]","Drama, Crime"
2,in the continuing sage of the corleone crime f...,"[18, 80]","Drama, Crime"
3,the true story of how businessman oscar schind...,"[18, 36, 10752]","Drama, History, War"
4,the defense and the prosecution have rested an...,[18],Drama
...,...,...,...
575,a young widow fleet from rome during wait and ...,"[18, 10752]","Drama, War"
576,tom road returns to his home after a jail sent...,[18],Drama
577,a photographer takes up newsreel shooting to i...,"[35, 10749]","Comedy, Romance"
578,when a machine that allows therapist to enter ...,"[878, 53, 16]","Science Fiction, Thrilled, Animation"


### Stop word removal

In [ ]:
nltk.download('stopwords')

stopwords = stopwords.words('english')
stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [ ]:
df['description'] = df['description'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stopwords)]))
df

,description,genre_ids,genres
0,imprisoned 1940s double murder wife lover stan...,"[18, 80]","Drama, Crime"
1,spanning years 1945 1955 chronicle sectional i...,"[18, 80]","Drama, Crime"
2,continuing sage corleone crime family young ve...,"[18, 80]","Drama, Crime"
3,true story businessman oscar schindler saved t...,"[18, 36, 10752]","Drama, History, War"
4,defense prosecution rested jury filing jury ro...,[18],Drama
...,...,...,...
575,young widow fleet rome wait takes lonely twelv...,"[18, 10752]","Drama, War"
576,tom road returns home jail sentence find famil...,[18],Drama
577,photographer takes newsreel shooting impress s...,"[35, 10749]","Comedy, Romance"
578,machine allows therapist enter patients dreams...,"[878, 53, 16]","Science Fiction, Thrilled, Animation"


### handling emojis

In [ ]:
df['description'] = df['description'].apply(lambda x: emoji.replace_emoji(x))
df


,description,genre_ids,genres
0,imprisoned 1940s double murder wife lover stan...,"[18, 80]","Drama, Crime"
1,spanning years 1945 1955 chronicle sectional i...,"[18, 80]","Drama, Crime"
2,continuing sage corleone crime family young ve...,"[18, 80]","Drama, Crime"
3,true story businessman oscar schindler saved t...,"[18, 36, 10752]","Drama, History, War"
4,defense prosecution rested jury filing jury ro...,[18],Drama
...,...,...,...
575,young widow fleet rome wait takes lonely twelv...,"[18, 10752]","Drama, War"
576,tom road returns home jail sentence find famil...,[18],Drama
577,photographer takes newsreel shooting impress s...,"[35, 10749]","Comedy, Romance"
578,machine allows therapist enter patients dreams...,"[878, 53, 16]","Science Fiction, Thrilled, Animation"


In [ ]:
emoji.replace_emoji('you have a cute face 😀')

'you have a cute face '

In [ ]:
emoji.demojize('you have a 😀')

'you have a :grinning_face:'

### Tokenisation

In [ ]:
tokenizer = spacy.load('en_core_web_sm')

In [ ]:

for doc in tokenizer('hello! Iam '):
  print(doc)

hello
!
Iam


In [ ]:
df['description'] = df['description'].apply(lambda x: [token.text for token in tokenizer(x)])
df

,description,genre_ids,genres
0,"[imprisoned, 1940s, double, murder, wife, love...","[18, 80]","Drama, Crime"
1,"[spanning, years, 1945, 1955, chronicle, secti...","[18, 80]","Drama, Crime"
2,"[continuing, sage, corleone, crime, family, yo...","[18, 80]","Drama, Crime"
3,"[true, story, businessman, oscar, schindler, s...","[18, 36, 10752]","Drama, History, War"
4,"[defense, prosecution, rested, jury, filing, j...",[18],Drama
...,...,...,...
575,"[young, widow, fleet, rome, wait, takes, lonel...","[18, 10752]","Drama, War"
576,"[tom, road, returns, home, jail, sentence, fin...",[18],Drama
577,"[photographer, takes, newsreel, shooting, impr...","[35, 10749]","Comedy, Romance"
578,"[machine, allows, therapist, enter, patients, ...","[878, 53, 16]","Science Fiction, Thrilled, Animation"


### Stemming and lemmatisation

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

df['stemmed_description'] = df['description'].apply(lambda x: [stemmer.stem(word) for word in x])
df['lemmatized_description'] = df['description'].apply(lambda x: [lemmatizer.lemmatize(word, pos='n') for word in x])
df

,description,genre_ids,genres,stemmed_description,lemmatized_description
0,"[imprisoned, 1940s, double, murder, wife, love...","[18, 80]","Drama, Crime","[imprison, 1940, doubl, murder, wife, lover, s...","[imprisoned, 1940s, double, murder, wife, love..."
1,"[spanning, years, 1945, 1955, chronicle, secti...","[18, 80]","Drama, Crime","[span, year, 1945, 1955, chronicl, section, it...","[spanning, year, 1945, 1955, chronicle, sectio..."
2,"[continuing, sage, corleone, crime, family, yo...","[18, 80]","Drama, Crime","[continu, sage, corleon, crime, famili, young,...","[continuing, sage, corleone, crime, family, yo..."
3,"[true, story, businessman, oscar, schindler, s...","[18, 36, 10752]","Drama, History, War","[true, stori, businessman, oscar, schindler, s...","[true, story, businessman, oscar, schindler, s..."
4,"[defense, prosecution, rested, jury, filing, j...",[18],Drama,"[defens, prosecut, rest, juri, file, juri, roo...","[defense, prosecution, rested, jury, filing, j..."
...,...,...,...,...,...
575,"[young, widow, fleet, rome, wait, takes, lonel...","[18, 10752]","Drama, War","[young, widow, fleet, rome, wait, take, lone, ...","[young, widow, fleet, rome, wait, take, lonely..."
576,"[tom, road, returns, home, jail, sentence, fin...",[18],Drama,"[tom, road, return, home, jail, sentenc, find,...","[tom, road, return, home, jail, sentence, find..."
577,"[photographer, takes, newsreel, shooting, impr...","[35, 10749]","Comedy, Romance","[photograph, take, newsreel, shoot, impress, s...","[photographer, take, newsreel, shooting, impre..."
578,"[machine, allows, therapist, enter, patients, ...","[878, 53, 16]","Science Fiction, Thrilled, Animation","[machin, allow, therapist, enter, patient, dre...","[machine, allows, therapist, enter, patient, d..."
